# SentinelDesk triage: train on Colab (from VS Code)

**How to connect (VS Code + Google Colab extension):** open this notebook → **Select Kernel** (top right) → **Colab** → **New Colab Server** → pick a **T4 GPU** runtime. Then run the cells top to bottom.

The Colab runtime is a remote machine: it does **not** see your laptop files. So we `git clone` the repo (push your branch first) and send the trained artifacts back at the end.

Rules that still apply here (see `CLAUDE.md`): the split comes only from `split.py`; nothing reads the holdout labels except `eval_triage.py`, and that runs **once, at the very end**.

In [1]:
import platform, os
print(platform.system(), platform.node(), os.getcwd())

Linux d1ddcba5aadc /content


In [2]:
# 1. Check the GPU (expect a Tesla T4). If this fails, switch the runtime to a GPU one.
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Get the code
Set `BRANCH` to the branch you pushed. The repo must be reachable from Colab (public, or use a token URL).

In [15]:
REPO_URL = "https://github.com/wuhong4088/HackathonGDG.git"
BRANCH = "shatru"
WORKDIR = "/content/HackathonGDG"

import os
if os.path.exists(WORKDIR):
    !git -C {WORKDIR} fetch origin {BRANCH} && git -C {WORKDIR} checkout {BRANCH} && git -C {WORKDIR} reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git log -1 --oneline

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 9.06 KiB | 2.26 MiB/s, done.
From https://github.com/wuhong4088/HackathonGDG
 * branch            shatru     -> FETCH_HEAD
   2d377e1..1eb3372  shatru     -> origin/shatru
Already on 'shatru'
Your branch is behind 'origin/shatru' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
HEAD is now at 1eb3372 triage update
/content/HackathonGDG
1eb3372 (HEAD -> shatru, origin/shatru) triage update


In [16]:
# 3. Install dependencies (keeps Colab's CUDA torch). scikit-learn must be 1.7.2 so the laptop can load the models.
!pip install -q -r requirements.txt
import sklearn, numpy, torch
print("sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert sklearn.__version__ == "1.7.2", "Restart the runtime (Runtime > Restart) and re-run from step 2"

sklearn 1.7.2 | numpy 2.1.3 | torch 2.11.0+cu130 | cuda True


## 4. Settings
* `EMBED_MODEL`: `intfloat/multilingual-e5-base` (default here: 768-dim, stronger than small; use a GPU), `intfloat/multilingual-e5-small` (fastest, fine on CPU) or `BAAI/bge-m3` (strongest, slowest). The live demo embeds each ticket with the same model (read from the artifacts).
* Artifacts go to `/content/artifacts` (local disk is fastest); step 9 copies them out.

In [17]:
EMBED_MODEL = "intfloat/multilingual-e5-base"    # or "intfloat/multilingual-e5-small" (fast) / "BAAI/bge-m3"
ARTIFACTS = "/content/artifacts"
os.environ["SENTINELDESK_ARTIFACTS"] = ARTIFACTS
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Optional: an HF token avoids Hub rate limits (dataset + model downloads).
# from getpass import getpass; os.environ["HF_TOKEN"] = getpass("HF token (optional): ")

In [18]:
# 5. Tests + a quick smoke run on 2,000 tickets (~1-2 min). Catches problems before the full run.
!python -m pytest -q tests/test_triage.py
!python scripts/build_split.py --sample 2000 && python scripts/embed.py --sample 2000 --model {EMBED_MODEL} && python scripts/train_triage.py --sample 2000 --model {EMBED_MODEL} | tail -5

...........s......                                                       [100%]
17 passed, 1 skipped in 2.26s
17:50:32 INFO sentineldesk.triage.data: kb=22869 test=5718  missing subjects in kb=3102
17:50:32 INFO sentineldesk.triage.data: type: ['Change', 'Incident', 'Problem', 'Request']
17:50:32 INFO sentineldesk.triage.data: queue: ['Billing and Payments', 'Customer Service', 'General Inquiry', 'Human Resources', 'IT Support', 'Product Support', 'Returns and Exchanges', 'Sales and Pre-Sales', 'Service Outages and Maintenance', 'Technical Support']
17:50:32 INFO sentineldesk.triage.data: priority: ['high', 'low', 'medium']
17:50:32 INFO sentineldesk.triage.data: wrote splits to /content/artifacts/sample_2000/splits: {'train': 2000, 'half_a': 111, 'half_b': 111, 'test': 556}
17:50:44 INFO datasets: TensorFlow version 2.21.0 available.
17:50:44 INFO datasets: JAX version 0.11.1 available.
17:50:46 INFO httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-base/resolv

## 6. Full training
Expected on a T4: split <1 min, embeddings ~1-2 min (e5-small) / ~3-5 min (e5-base) / ~5-8 min (bge-m3); on a CPU runtime e5-base takes ~1 h, training ~3-6 min.

In [19]:
%%time
!python scripts/build_split.py

17:51:13 INFO sentineldesk.triage.data: kb=22869 test=5718  missing subjects in kb=3102
17:51:13 INFO sentineldesk.triage.data: type: ['Change', 'Incident', 'Problem', 'Request']
17:51:13 INFO sentineldesk.triage.data: queue: ['Billing and Payments', 'Customer Service', 'General Inquiry', 'Human Resources', 'IT Support', 'Product Support', 'Returns and Exchanges', 'Sales and Pre-Sales', 'Service Outages and Maintenance', 'Technical Support']
17:51:13 INFO sentineldesk.triage.data: priority: ['high', 'low', 'medium']
17:51:13 INFO sentineldesk.triage.data: wrote splits to /content/artifacts/splits: {'train': 20582, 'half_a': 1144, 'half_b': 1143, 'test': 5718}
CPU times: user 6.37 ms, sys: 2.06 ms, total: 8.43 ms
Wall time: 1.41 s


In [8]:
%%time
!python scripts/embed.py --model {EMBED_MODEL}

17:32:33 INFO datasets: TensorFlow version 2.21.0 available.
17:32:33 INFO datasets: JAX version 0.11.1 available.
17:32:35 INFO httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
17:32:35 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
17:32:35 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-small/614241f622f53c4eeff9890bdc4f31cfecc418b3/modules.json?%2Fintfloat%2Fmultilingual-e5-small%2Fresolve%2Fmain%2Fmodules.json=&etag=%22ac2039abdf6ff023b27c919bb9675cfe378cb10f%22 "HTTP/1.1 200 OK"
17:32:35 INFO httpx: HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
17:32:35 INFO sentence_transformers.base.model: Loading SentenceTrans

In [20]:
%%time
# Trains 0, 1, 5 on train; ensemble on half A; temperature + thresholds on half B. Prints half-B metrics.
!python scripts/train_triage.py --model {EMBED_MODEL}

No embeddings for intfloat/multilingual-e5-base in /content/artifacts/embeddings/intfloat__multilingual-e5-base.
Run first:  python scripts/embed.py --model intfloat/multilingual-e5-base
Embeddings that exist: ['intfloat__multilingual-e5-small']
CPU times: user 6.87 ms, sys: 1.22 ms, total: 8.09 ms
Wall time: 1.61 s


In [21]:
# 7. Sanity check: load the trained Triage and predict two live tickets (English + German, one without subject).
import sys, json; sys.path.insert(0, WORKDIR)
from sentineldesk.triage.predict import Triage
triage = Triage.load(ARTIFACTS)
for subj, body in [("VPN outage", "Since 8am nobody in the clinic can reach the EMR system, all users affected."),
                   (None, "Guten Tag, ich habe eine Frage zu meiner letzten Rechnung. Können Sie mir eine Kopie schicken?")]:
    r = triage.predict(subj, body)
    print(json.dumps({k: r[k] for k in ("mode", "critical", "latency_ms")} |
                     {t: (r[t]["label"], r[t]["confidence"]) for t in ("type", "queue", "priority")}, ensure_ascii=False, indent=1))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{
 "mode": "full",
 "critical": {
  "score": 0.8685,
  "threshold": 0.4724,
  "fires": true,
  "keyword_hits": [
   "all users",
   "emr",
   "outage"
  ]
 },
 "latency_ms": {
  "embed": 11.84,
  "models": 3.1,
  "neighbours": 3.24,
  "tfidf": 1.94,
  "ensemble": 0.66,
  "critical": 0.36,
  "total": 21.26
 },
 "type": [
  "Incident",
  0.9123
 ],
 "queue": [
  "Technical Support",
  0.2609
 ],
 "priority": [
  "medium",
  0.4833
 ]
}
{
 "mode": "full",
 "critical": {
  "score": 0.0559,
  "threshold": 0.4724,
  "fires": false,
  "keyword_hits": []
 },
 "latency_ms": {
  "embed": 11.85,
  "models": 2.81,
  "neighbours": 3.12,
  "tfidf": 1.75,
  "ensemble": 0.64,
  "critical": 0.4,
  "total": 20.64
 },
 "type": [
  "Incident",
  0.7065
 ],
 "queue": [
  "Billing and Payments",
  0.9549
 ],
 "priority": [
  "medium",
  0.4266
 ]
}


## 8. Holdout evaluation (ONCE, at the very end)
Only run this when the models are final. Never change keywords, thresholds or hyperparameters after looking at these numbers.

In [ ]:
RUN_HOLDOUT_EVAL = False   # set True only for the final run
if RUN_HOLDOUT_EVAL:
    !python scripts/eval_triage.py
    !mkdir -p {ARTIFACTS}/results && cp -r results/triage_metrics.json results/plots {ARTIFACTS}/results/

## 9. Send the artifacts back to your laptop
The VS Code Colab extension does not support `drive.mount()` or `files.download()` reliably, so try these in order:

1. **Google Drive** (works if mounting succeeds in your extension version)
2. **Private Hugging Face repo** (most reliable; needs an HF *write* token)
3. Fall back to the Colab file browser / "Download" on `/content/sentineldesk_artifacts.zip`

The zip holds `triage/` (all the demo needs, self-contained), `embeddings/` (reusable by the retrieval teammate) and `splits/`.

In [ ]:
# Build the zip (skips quick-run sample folders).
!cd {ARTIFACTS} && rm -f /content/sentineldesk_artifacts.zip && zip -qr /content/sentineldesk_artifacts.zip . -x "sample_*" && ls -lh /content/sentineldesk_artifacts.zip

In [ ]:
# Option 1: Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/sentineldesk && cp /content/sentineldesk_artifacts.zip /content/drive/MyDrive/sentineldesk/
    print("Copied to My Drive/sentineldesk/sentineldesk_artifacts.zip")
except Exception as e:
    print("Drive mount not available here:", e, "-> use option 2")

In [ ]:
# Option 2: private Hugging Face model repo (artifacts derive from a CC-BY-NC dataset: keep it private).
from getpass import getpass
from huggingface_hub import HfApi
HF_REPO = "YOUR_HF_USERNAME/sentineldesk-artifacts"   # <- change
api = HfApi(token=getpass("HF write token: "))
api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_file(path_or_fileobj="/content/sentineldesk_artifacts.zip", path_in_repo="sentineldesk_artifacts.zip",
                repo_id=HF_REPO, repo_type="model")
print(f"Uploaded. On the laptop: hf download {HF_REPO} sentineldesk_artifacts.zip --local-dir .")

## On the laptop
```bash
# from the repo root, with the zip downloaded next to it
mkdir -p artifacts && unzip -o sentineldesk_artifacts.zip -d artifacts
python -c "from sentineldesk.triage.predict import Triage; print(Triage.load('artifacts').predict(None, 'VPN down for all users')['critical'])"
```